In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

ollama_client = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")
msg = ollama_client.chat.completions.create(model="gemma3:4b",messages=[{'role':'user','content':'Hi'}])
msg.choices[0].message.content

In [ ]:
questions = """Questions:
Q1: What is the capital of France?
A. Berlin
B. Madrid
C. Paris
D. Rome

Q2: What is 2 + 2?
A. 3
B. 4
C. 5
D. 6

Q3: Which color is the sky on a clear day?
A. Blue
B. Green
C. Red
D. Yellow"""
questions_arr = questions.split("\n\n")
questions_arr

In [ ]:
import json
answers= """Student: Alice
Q1: C
Q2: B
Q3: A

Student: Bob
Q1: A
Q2: B
Q3: A

Student: Charlie
Q1: C
Q2: D
Q3: A"""

anwers_arr = answers.split("\n\n")


In [ ]:
def show_student_completed(student_name, score, total_questions):
    print("---------------------------")
    print(f"Student: {student_name}")
    print(f"Score: {score}/{total_questions}")
    print("Status: Completed")
    print("---------------------------")
    return {"info":'shared'}

In [ ]:


def graded(feedback: str, student_name: str) -> str:
    print("---------------------------")
    print(f"Student: {student_name}")
    print(f"Result : {feedback}")
    print("---------------------------")
    return {"info":'feed back provided'}

def save_student_report(student_name: str, score: int, total_questions: int, feedback: str) -> dict:
    with open("student_report.txt", "a", encoding="utf-8") as f:
        f.write(f"Student: {student_name}\n")
        f.write(f"Score: {score}/{total_questions}\n")
        f.write(f"Feedback: {feedback}\n")
        f.write("-" * 30 + "\n")
    return {"info": "student report saved"}

def read_student_report():
    with open("student_report.txt","r",encoding="utf-8") as f:
        data = f.read()
    return data
    
save_student_report_tool = {
    "name": "save_student_report",
    "description": "Save one student's result and feedback to a local text file.",
    "parameters": {
        "type": "object",
        "properties": {
            "student_name": {
                "type": "string"
            },
            "score": {
                "type": "integer"
            },
            "total_questions": {
                "type": "integer"
            },
            "feedback": {
                "type": "string"
            }
        },
        "required": ["student_name", "score", "total_questions", "feedback"]
    }
}

read_student_report_tool = {
    "name": "read_student_report",
    "description": "Reeds and provides the student's feedback report which have been updated till now.",
    "parameters": {
        "type": "object",
        "properties": {
        },
        "required": []
    }
}

show_student_completed_tool = {
    "name": "show_student_completed",
    "description":"use this function after you have evaluated one student.",
    "parameters":{
        'type':'object',
        "properties":{
            'student_name':{
                'type':'string',
                'description':'the name of the student'
            },
            'score':{
                'type':'integer',
                'description':'total number of questions. The student got it right.'
            },
            'total_questions':{
                'type':'integer',
                'description':'total number of courses that are there'
            }
        }
    },
    'required': ['student_name','score','total_questions']
}

graded_tool = {
    "name": "graded",
    "description":"Use this function after you have evaluated each student that have been provided.",
    "parameters":{
        'type':'object',
        "properties":{
            'feedback':{
                'type':'string',
                'description':'Feedback about each student on why they might have thought that is the answer and the correct response'
            },
            'student_name':{
                'type':'string',
                'description':'The name of the student.'
            }
        }
    },
    'required': ['student_name','score','total_questions']
}


In [ ]:
tools = [{'type':'function','function':show_student_completed_tool},
{'type':'function','function':graded_tool},{'type':'function','function':save_student_report_tool},{'type':'function','function':read_student_report_tool}]

def handle_tool_calls(tool_calls):
    results=[]
    for tool_call in tool_calls:
        print(f"Tool Called {tool_call.function.name}")
        arguments = json.loads(tool_call.function.arguments)
        result = globals().get(tool_call.function.name)(**arguments) 

        results.append({"role":'tool','content':json.dumps(result),'tool_call_id': tool_call.id})
    return results


from openai import OpenAI
import os
openai = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))     
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content


In [ ]:


def messages_for(answers):
    total_students = len(answers)
    for i,answer in enumerate(answers):

        system_prompt = """ 
        You are a quiz evaluation assistant.

        Your job is to evaluate student answers question by question and student by student.

        You will receive:
        1. A list of questions
        2. A list of correct answers
        3. A student name
        4. A list of that student's chosen answers

        Your tasks:
        - Compare the student's chosen answers with the correct answers
        - Calculate the student's score
        - For each question, say whether it is correct or incorrect
        - If incorrect, briefly explain the correct answer
        - Give a short overall feedback summary for the student
        - After feedback, you have to write that into a file using the tool and show the progress also by using tool after each student (show_student_completed_tool)
        - Keep the response clear, structured, and concise

        Rules:
        - Treat each answer by index position
        - Question 1 matches answer index 0
        - Question 2 matches answer index 1
        - Do not invent extra questions or answers
        - If the number of student answers does not match the number of correct answers, report the mismatch clearly
        - If you are given any info that student is the last student, you are checking the answers then, after providing the feedback for the current student and writing it into the file which is provided, you need to make an overall feedback of how students for this are. Also, you can use a tool to read it from the file you have written.  
        - Output in plain text
        - When you are done with the student, don't call any tool. Just complete by a response, saying "Once the student evaluation of the student name is completed."
        """
        last_student="No"
        if total_students == i+1:
            last_student = "Yes" 
        user_prompt = f""" 
        Evaluate this student's quiz.

        Student name and answers: 
        Student answers:
        {answer}

        Questions:
        {questions}

        Correct answers:
        ["C", "B", "A"]

        
        isLastStudent = {last_student}
        """
        messages = [{'role':'system','content':system_prompt},{'role':'user','content':user_prompt}]
        resp = loop(messages)

resp = messages_for(anwers_arr)



In [ ]:
for i,answer in enumerate(anwers_arr):
    print(anwers_arr[i])
